In [1]:
import pandas as pd

anime = pd.read_csv('anime.csv', low_memory=False)
rating = pd.read_csv('rating_complete.csv', low_memory=False)
synopsis = pd.read_csv('anime_with_synopsis.csv', low_memory=False)

In [2]:
anime = anime.rename(columns={'MAL_ID': 'anime_id'})
synopsis = synopsis.rename(columns={'MAL_ID': 'anime_id'})

In [3]:
anime_rating = pd.DataFrame(rating.groupby('anime_id').agg(rating = ('rating', 'mean'), vote = ('rating', 'sum'))).reset_index()
anime_rating.head()

,anime_id,rating,vote
0,1,8.763003,698534
1,5,8.318865,279622
2,6,8.243815,387212
3,7,7.409233,75752
4,8,7.150594,10826


In [4]:
anime_rating['rating'] = anime_rating['rating'].fillna(0)
anime_rating.describe()

,anime_id,rating,vote
count,16872.000000,16872.000000,1.687200e+04
mean,20701.459104,6.196639,2.565620e+04
std,14629.802658,1.118052,8.239312e+04
min,1.000000,1.749910,4.000000e+00
25%,5599.250000,5.468604,2.080000e+02
50%,21438.000000,6.288750,1.558500e+03
75%,34961.250000,7.000000,1.108175e+04
max,48456.000000,10.000000,1.587860e+06


In [5]:
anime_rating.sort_values(by=['vote'], ascending=True)

,anime_id,rating,vote
16853,45612,4.000000,4
15403,39436,4.000000,4
13149,35746,4.000000,4
16300,41416,4.000000,4
16264,41277,4.000000,4
...,...,...,...
6603,11757,7.403593,1193400
1428,1575,8.768004,1203768
3965,5114,9.237010,1239579
7437,16498,8.514506,1445712


In [6]:
len(anime_rating)

16872

In [7]:
m = anime_rating['vote'].quantile(0.9)
m

np.float64(61600.89999999996)

In [8]:
qualified_anime = anime_rating.copy().loc[anime_rating['vote'] >= m]
len(qualified_anime)

1688

In [9]:
rating = rating[rating['anime_id'].isin(qualified_anime['anime_id'])]
len(rating)

43555008

In [10]:
user_anime_matrix = rating.pivot_table(index='user_id', columns='anime_id', values='rating').fillna(0)
user_anime_matrix.head()

anime_id,1,5,6,7,15,16,18,19,20,22,...,40956,41120,41168,41226,41353,41389,41433,41468,41619,41930
user_id,,,,,,,,,,,,,,,,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,10.0,9.0,...,7.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,9.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,9.0,0.0,0.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
def get_recommendations(anime_name):
  matches = anime.loc[anime['Name'] == anime_name, 'anime_id'].values
  if len(matches) > 0:
    anime_id = matches[0]
  else:
    print('Anime not found')
    return

  users_ratings = user_anime_matrix[anime_id]
  similar_animes = user_anime_matrix.corrwith(users_ratings)

  similar_animes = pd.DataFrame(similar_animes, columns=['correlation'])

  df_name = anime[['anime_id', 'Name']]
  df_name = df_name[df_name['anime_id'].isin(similar_animes.index)]
  df_synopsis = synopsis[['anime_id', 'sypnopsis']]
  df_synopsis = df_synopsis[df_synopsis['anime_id'].isin(similar_animes.index)]
  df_anime = df_name.merge(df_synopsis, on='anime_id')
  similar_animes_with_name = similar_animes.merge(df_anime, on='anime_id').sort_values(by=['correlation'], ascending=False)

  print(similar_animes_with_name.head(10))

In [12]:
anime_id = anime[anime['Name'] == "Naruto"]['anime_id'].values[0]
users_ratings = user_anime_matrix[anime_id]
similar_animes = user_anime_matrix.corrwith(users_ratings)
similar_animes = pd.DataFrame(similar_animes, columns=['correlation']).reset_index()
similar_animes.sort_values(by=['correlation'], ascending=False)

,anime_id,correlation
8,20,1.000000
346,1735,0.516318
386,2472,0.362631
162,442,0.359324
126,269,0.358000
...,...,...
766,10495,-0.013119
1017,18617,-0.014153
886,14131,-0.015930
273,949,-0.016000


In [13]:
df_name = anime[['anime_id', 'Name']]
df_name = df_name[df_name['anime_id'].isin(similar_animes.index)]
df_synopsis = synopsis[['anime_id', 'sypnopsis']]
df_synopsis = df_synopsis[df_synopsis['anime_id'].isin(similar_animes.index)]
df_anime = df_name.merge(df_synopsis, on='anime_id')
df_anime.head()

,anime_id,Name,sypnopsis
0,1,Cowboy Bebop,"In the year 2071, humanity has colonized sever..."
1,5,Cowboy Bebop: Tengoku no Tobira,"other day, another bounty—such is the life of ..."
2,6,Trigun,"Vash the Stampede is the man with a $$60,000,0..."
3,7,Witch Hunter Robin,ches are individuals with special powers like ...
4,8,Bouken Ou Beet,It is the dark century and the people are suff...


In [14]:
similar_animes_with_name = similar_animes.merge(df_anime, on='anime_id').sort_values(by=['correlation'], ascending=False)
similar_animes_with_name.head()

,anime_id,correlation,Name,sypnopsis
8,20,1.000000,Naruto,"oments prior to Naruto Uzumaki's birth, a huge..."
162,442,0.359324,Naruto Movie 1: Dai Katsugeki!! Yuki Hime Shin...,"Naruto Uzumaki and his squadmates, Sasuke Uchi..."
126,269,0.358000,Bleach,Ichigo Kurosaki is an ordinary high schooler—u...
271,936,0.335939,Naruto Movie 2: Dai Gekitotsu! Maboroshi no Ch...,"Naruto, Shikamaru, and Sakura are executing th..."
239,813,0.304259,Dragon Ball Z,Five years after winning the World Martial Art...


In [15]:
user_input = input('Enter anime name that you want to get similar rating anime: ')
get_recommendations(user_input)

Enter anime name that you want to get similar rating anime:  Cowboy Bebop


     anime_id  correlation                             Name  \
0           1     1.000000                     Cowboy Bebop   
1           5     0.549773  Cowboy Bebop: Tengoku no Tobira   
102       205     0.456544                 Samurai Champloo   
2           6     0.426363                           Trigun   
108       227     0.397404                             FLCL   
22         47     0.394175                            Akira   
15         30     0.387036          Neon Genesis Evangelion   
19         43     0.385175                 Koukaku Kidoutai   
253       889     0.354376                     Black Lagoon   
380      2251     0.352991                         Baccano!   

                                             sypnopsis  
0    In the year 2071, humanity has colonized sever...  
1    other day, another bounty—such is the life of ...  
102  Fuu Kasumi is a young and clumsy waitress who ...  
2    Vash the Stampede is the man with a $$60,000,0...  
108  Naota Nandaba is